In [3]:
# coding: utf-8

import torch
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import numpy as np
import os
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import sys
sys.path.append("..")
from utils.utils import MyDataset, validate, show_confMat
from datetime import datetime

train_txt_path = os.path.join("..", "..", "Data", "train.txt")
valid_txt_path = os.path.join("..", "..", "Data", "valid.txt")

classes_name = ['plane', 'car', 'bird', 'cat', 'deer', 'dog',\
                'frog', 'horse', 'ship', 'truck']

train_bs = 16
valid_bs = 16
lr_init = 0.001
max_epoch = 1

# log
result_dir = os.path.join("..", "..", "Result")

now_time = datetime.now()
time_str = datetime.strftime(now_time, '%m-%d_%H-%M-%S')

log_dir = os.path.join(result_dir, time_str)
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

# -------------------------------------------- step 1/5 : 加载数据 -------------------------------------------

# 数据预处理设置
normMean = [0.4948052, 0.48568845, 0.44682974]
normStd = [0.24580306, 0.24236229, 0.2603115]
normTransform = transforms.Normalize(normMean, normStd)
trainTransform = transforms.Compose([
    transforms.Resize(32),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    normTransform
])

validTransform = transforms.Compose([
    transforms.ToTensor(),
    normTransform
])

# 构建MyDataset实例
# train_data = MyDataset(txt_path=train_txt_path, transform=trainTransform)
# valid_data = MyDataset(txt_path=valid_txt_path, transform=validTransform)

# # 构建DataLoder
# train_loader = DataLoader(dataset=train_data, batch_size=train_bs, shuffle=True)
# valid_loader = DataLoader(dataset=valid_data, batch_size=valid_bs)

# ------------------------------------ step 2/5 : 定义网络 ------------------------------------


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.initialize_weights()
        
    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    # 定义权值初始化
    def initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                torch.nn.init.xavier_normal_(m.weight.data)
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight.data, 0, 0.01)
                m.bias.data.zero_()


net = Net()     # 创建一个网络

# ================================ #
#        finetune 权值初始化
# ================================ #

f:\my_softers\project_IDE\Anconda\envs\jp_layout_pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
net.modules()

<generator object Module.modules at 0x0000022492066DC8>

In [19]:
a[1]

Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))

In [5]:
a = list(net.modules())
print(a)
print('='*6)
print(isinstance(a[1],nn.Conv2d)) # True
print(a[1].weight.requires_grad)  # requires_grad = True
print(a[1].weight.data.requires_grad) # False
print(a[1].weight.detach().requires_grad) # False
print(id(a[1].weight))
print(id(a[1].weight.data))


[Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
), Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1)), MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False), Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1)), MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False), Linear(in_features=400, out_features=120, bias=True), Linear(in_features=120, out_features=84, bias=True), Linear(in_features=84, out_features=10, bias=True)]
True
True
False
False
2356092314872
2355904560168


In [6]:
# 从权重文件中获取的 网络层级结构和对应的权重
pretrained_dict = torch.load('net_params.pkl')
# for key, val in pretrained_dict.items():
#     print(f'key:{key},val:{val.shape}')
pretrained_dict.keys()

odict_keys(['conv1.weight', 'conv1.bias', 'conv2.weight', 'conv2.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias'])

In [7]:
list(pretrained_dict.values())[0].shape

torch.Size([6, 3, 5, 5])

In [8]:
pretrained_dict['conv1.weight'].shape

torch.Size([6, 3, 5, 5])

In [9]:
[ x.shape for x in pretrained_dict.values() ]

[torch.Size([6, 3, 5, 5]),
 torch.Size([6]),
 torch.Size([16, 6, 5, 5]),
 torch.Size([16]),
 torch.Size([120, 400]),
 torch.Size([120]),
 torch.Size([84, 120]),
 torch.Size([84]),
 torch.Size([10, 84]),
 torch.Size([10])]

In [10]:
# 从构建的网络中获取 网络层级结构和对应的初始化权重参数
net_state_dict = net.state_dict()
net_state_dict.keys()

odict_keys(['conv1.weight', 'conv1.bias', 'conv2.weight', 'conv2.bias', 'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias'])

In [11]:
[ x.shape for x in net_state_dict.values() ]

[torch.Size([6, 3, 5, 5]),
 torch.Size([6]),
 torch.Size([16, 6, 5, 5]),
 torch.Size([16]),
 torch.Size([120, 400]),
 torch.Size([120]),
 torch.Size([84, 120]),
 torch.Size([84]),
 torch.Size([10, 84]),
 torch.Size([10])]

In [12]:
# 插播一条字典的使用方法
dict_test = {'a':1,'b':2,'c':3}
updata = {'a':111, 'd':444}
dict_test.update(updata) # 修改已有的key的内容， 添加未知的key
dict_test

{'a': 111, 'b': 2, 'c': 3, 'd': 444}

In [13]:
# 将初始化权重中的参数 更新为训练参数中对应的参数值
pretrained_dict1 = {key:val for key,val in pretrained_dict.items() if key in net_state_dict.keys()}
net_state_dict.update(pretrained_dict1)

net.load_state_dict(net_state_dict)

<All keys matched successfully>

<span style='color:red'>1. model.state_dict()与model.parameters()的区别  
a. model.state_dict()包含网络层的名称和对应的参数-字典={'层1名称':'对应的权重',...}--常用于加载权重    
&emsp;model.parameters()仅仅包含对应的参数  
&emsp;model.named_parameters()包含网络层的名称和对应的参数-元组=((层1名称,对应的权重),...)
</span>

In [14]:
[ x.shape for x in net.state_dict().values() ]

[torch.Size([6, 3, 5, 5]),
 torch.Size([6]),
 torch.Size([16, 6, 5, 5]),
 torch.Size([16]),
 torch.Size([120, 400]),
 torch.Size([120]),
 torch.Size([84, 120]),
 torch.Size([84]),
 torch.Size([10, 84]),
 torch.Size([10])]

In [15]:
net.parameters() # net网络所有网络层的参数
# net.指定层的名称.parameters() # net网络指定层名对应的网络层的参数
[ x.shape for x in net.parameters() ]

[torch.Size([6, 3, 5, 5]),
 torch.Size([6]),
 torch.Size([16, 6, 5, 5]),
 torch.Size([16]),
 torch.Size([120, 400]),
 torch.Size([120]),
 torch.Size([84, 120]),
 torch.Size([84]),
 torch.Size([10, 84]),
 torch.Size([10])]

In [16]:
for name, data in net.named_parameters():
    print('name:',name)
    print(data.shape)

name: conv1.weight
torch.Size([6, 3, 5, 5])
name: conv1.bias
torch.Size([6])
name: conv2.weight
torch.Size([16, 6, 5, 5])
name: conv2.bias
torch.Size([16])
name: fc1.weight
torch.Size([120, 400])
name: fc1.bias
torch.Size([120])
name: fc2.weight
torch.Size([84, 120])
name: fc2.bias
torch.Size([84])
name: fc3.weight
torch.Size([10, 84])
name: fc3.bias
torch.Size([10])


In [17]:
import numpy as np
np.array(list(net.conv1.parameters()))[0].shape

torch.Size([6, 3, 5, 5])

In [18]:
[x for x in net]

TypeError: 'Net' object is not iterable

In [ ]:
net.conv2.parameters()

<generator object Module.parameters at 0x000002BBD251CDC8>

In [ ]:
a = 1
b = 1 
print(f'{a:.3f}')  # 保留3位小数位
print(f'{b:0>3}')  # 用0填充，该字符要占3位，右对齐
print(f'{b:0<3}')  # 用0填充，该字符要占3位，左对齐

1.000
001
100


####### pytorch 保存与加载模型 #######

In [ ]:
### -------------- 常规操作 ------------ ###
# -------- 整个模型的保存与加载 ---------
# a. 模型保存
model = VGGNet()
torch.save(model,'***.pt')
# b. 模型加载
model = torch.load(Path)

# --------- 仅仅保存模型的参数 ----------
# a. 模型保存
model = VGGNet()
torch.save(model.state_dict(),'***.pt')
# b. 模型加载
model.load_state_dict(torch.load(Path))


In [ ]:
### 将模型参数和优化器参数中的参数都保存起来 ###
## -- 保存
checkpoint = {
    "model_state_dict":net.state_dict(),
    "optimizer_state_dict":optimizer.state_dict(),
    "epoch":epoch,
    "lr":optimizer.param_groups[0][lr]    
}
path_checkpoint = "./checkpoint_{}_epoch.pkl".format(epoch)
torch.save(checkpoint, path_checkpoint)
## -- 加载

checkpoint = torch.load(path_checkpoint)
# 恢复模型参数
net.load_state_dict(checkpoint["model_state_dict"])
# 恢复优化器参数
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# 设置要恢复的epoch
start_epoch = checkpoint["epoch"]
# 设置要恢复的学习率
start_epoch = checkpoint["lr"]


In [ ]:
### -------------- ONNX格式 ------------ ###
import onnx
import onnxruntime

# a. 模型保存
model = VGGNet()
input_sample = torch.randn(16,128) # 提供一个输入样本作为示例
torch.onnx.export(model,input_sample,'sample_model.onnx')
# b. 模型加载
model = onnx.load('sample_model.onnx')
session = onnxruntime.InferenceSession('sample_model.onnx')
print(session)

In [ ]:
# 参考：
# https://blog.csdn.net/XDH19910113/article/details/127306806
##  保存模型时遇到的多卡问题
# 当多卡训练保存的参数名称：module.conv1.weight
# 而单卡训练保存的参数名称：conv1.weight
# 这是当你保存模型后，在重新加载就会报错，找不到对应的字典

## 解决方法1：
# 将 torch.save(model.state_dict(),'fff.pt')
#                |
#               \ /
#    torch.save(model.module.state_dict(),'fff.pt')
    
    

In [ ]:
# 使用 torch.jit(为了后面工程化使用C++调用该模型)
resnet18 = models.resnet34(pretrained=True)
dummy_input = torch.rand(1,3,256,256).to('cpu')
torch_trace_model = torch.jit.trace(resnet18,dummy_input)
torch.jit.save(torch_trace_model,'xxx.pth')






